# Week 3 Task — Unsupervised Learning and Clustering Analysis
**Approach:** RFM (Recency, Frequency, Monetary) feature engineering + K-Means clustering
**Input:** `data/online_retail_cleaned.csv` (Week 1 output)

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/online_retail_cleaned.csv"

df = pd.read_csv(DATA_PATH)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

if df["IsCancellation"].dtype != bool:
    df["IsCancellation"] = (
        df["IsCancellation"].astype(str).str.strip().str.lower()
        .map({"true": True, "1": True, "1.0": True,
              "false": False, "0": False, "0.0": False})
        .fillna(False).astype(bool)
    )

sales = df[~df["IsCancellation"]].dropna(subset=["Customer ID"]).copy()
sales.head()

## 2. RFM Feature Engineering

For each customer: Recency (days since last order), Frequency (distinct
orders), Monetary (total spend).

In [ ]:
ref_date = sales["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Reference date:", ref_date)

rfm = sales.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (ref_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("TotalAmount", "sum"),
).reset_index()

print(f"Customers: {rfm.shape[0]:,}")
rfm.describe()

## 3. Preprocessing

Frequency and Monetary are right-skewed — apply log1p, then standardize
all three features.

In [ ]:
rfm_log = rfm.copy()
rfm_log["Frequency"] = np.log1p(rfm_log["Frequency"])
rfm_log["Monetary"] = np.log1p(rfm_log["Monetary"])

X = rfm_log[["Recency", "Frequency", "Monetary"]]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 4. Choosing k — Elbow Method and Silhouette Score

In [ ]:
inertias, sils = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(list(K_range), inertias, marker="o", color="#2E86AB", label="Inertia (WCSS)")
ax1.set_xlabel("Number of Clusters (k)"); ax1.set_ylabel("Inertia (WCSS)", color="#2E86AB")
ax2 = ax1.twinx()
ax2.plot(list(K_range), sils, marker="s", color="#A23B72", label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color="#A23B72")
ax1.set_title("Elbow Method & Silhouette Score for Optimal k", fontsize=13, fontweight="bold")
ax1.axvline(4, color="gray", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

for k, i, s in zip(K_range, inertias, sils):
    print(f"k={k}: inertia={i:.1f}  silhouette={s:.3f}")

**Choice:** k=4 sits at the elbow bend and gives four business-interpretable
segments, even though silhouette is technically highest at k=2/3 (which mostly
just splits active vs. inactive customers).

## 5. Fit Final Model (k=4)

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(X_scaled)
rfm["Cluster"].value_counts().sort_index()

## 6. Cluster Visualization (PCA)

In [ ]:
pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_scaled)
rfm["PC1"], rfm["PC2"] = pcs[:, 0], pcs[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=rfm, x="PC1", y="PC2", hue="Cluster", palette="Set2", alpha=0.6, s=25, ax=ax)
ax.set_title(f"Customer Segments in PCA Space (k={k})\n"
             f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Cluster Profiles

In [ ]:
profile = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(1)
profile["Count"] = rfm["Cluster"].value_counts().sort_index()
profile["Share"] = (profile["Count"] / profile["Count"].sum() * 100).round(1)
profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, col, color in zip(axes, ["Recency", "Frequency", "Monetary"], ["#2E86AB", "#A23B72", "#F18F01"]):
    sns.barplot(x=profile.index, y=profile[col], ax=ax, color=color)
    ax.set_title(f"Avg {col} by Cluster", fontsize=11, fontweight="bold")
    ax.set_xlabel("Cluster")
plt.tight_layout()
plt.show()

## 8. Segment Labels

Based on the profile table above:

| Cluster | Segment | Recency | Frequency | Monetary |
|---|---|---|---|---|
| 2 | **Champions** | Very recent | High | Very high |
| 3 | **Loyal Customers** | Recent | Moderate | Moderate-high |
| 0 | **New / Occasional** | Recent | Low | Low |
| 1 | **At Risk / Lapsed** | Long ago | Low | Low |

- **Champions** (~15%): recent, frequent, high-value — prioritize for retention.
- **Loyal Customers** (~32%): the largest revenue-generating middle tier — target for upsell.
- **New / Occasional** (~25%): recent but low engagement — nurture toward a second purchase.
- **At Risk / Lapsed** (~28%): haven't ordered in over a year — win-back candidates.

In [ ]:
rfm.to_csv("rfm_clustered.csv", index=False)
rfm.head()